# Лабораторна робота №2

## Аналіз даних та прогнозування енергоспоживання

Варіант 3: лабораторія, Київ, 2025. Notebook виконує повний batch-аналіз на основі SQLite-бази, створеної у ЛР1.

## 1. Імпорт бібліотек і шляхи проєкту

In [ ]:
from pathlib import Path
import sqlite3
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
DB_PATH = ROOT / 'data' / 'energy_lab1_v3.sqlite'
FIG_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR = ROOT / 'outputs' / 'tables'
print('ROOT =', ROOT)
print('DB exists =', DB_PATH.exists())

## 2. Завантаження даних з ЛР1

Дані беруться не з ручного CSV, а з бази `data/energy_lab1_v3.sqlite`, сформованої у першій лабораторній роботі.

In [ ]:
query = '''
SELECT
    ms.timestamp,
    ms.consumption_kwh,
    w.temperature_c,
    w.insolation_kw_m2,
    w.hdd_18,
    w.cdd_22,
    b.baseline_kwh,
    ts.zone_name,
    ts.import_price_uah_kwh
FROM measurements ms
JOIN meters m ON m.meter_id = ms.meter_id AND m.accounting_level = 1
JOIN weather_data w ON w.object_id = m.object_id AND w.timestamp = ms.timestamp
JOIN baselines b ON b.object_id = m.object_id AND b.timestamp = ms.timestamp
JOIN tariff_schedule ts ON ts.hour = CAST(STRFTIME('%H', ms.timestamp) AS INTEGER)
ORDER BY ms.timestamp
'''
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(query, conn, parse_dates=['timestamp'])

df.head()

In [ ]:
print('Кількість рядків:', len(df))
print('Період:', df['timestamp'].min(), '—', df['timestamp'].max())
print('Пропуски:', int(df.isna().sum().sum()))
print('Дублікати часу:', int(df['timestamp'].duplicated().sum()))

## 3. Описові статистики та якість даних

In [ ]:
desc = df[['consumption_kwh','temperature_c','insolation_kw_m2','hdd_18','cdd_22','baseline_kwh']].describe().T
desc

In [ ]:
diagnostics = pd.DataFrame({
    'metric': ['rows', 'missing_values', 'duplicated_timestamps', 'outliers_z_gt_3'],
    'value': [
        len(df),
        int(df.isna().sum().sum()),
        int(df['timestamp'].duplicated().sum()),
        int(((df['consumption_kwh'] - df['consumption_kwh'].mean()).abs() > 3 * df['consumption_kwh'].std()).sum()),
    ],
})
diagnostics

## 4. Дослідницький аналіз даних: візуалізації

Нижче відображаються всі графіки, які генерує `src/run_analysis.py`. Якщо їх немає, спочатку виконайте клітинку з повним запуском аналізу.

In [ ]:
# Повний запуск pipeline: EDA, таблиці, моделі, прогноз.
# У разі повторного запуску всі результати у outputs/ оновлюються.
%run ../src/run_analysis.py

### Річна динаміка добового споживання

In [ ]:
display(Image(filename=str(FIG_DIR / '01_year_daily_consumption.png')))

### Гістограма розподілу погодинного споживання

In [ ]:
display(Image(filename=str(FIG_DIR / '02_consumption_histogram.png')))

### Box plot споживання за місяцями

In [ ]:
display(Image(filename=str(FIG_DIR / '03_monthly_boxplot.png')))

### Типовий добовий профіль

In [ ]:
display(Image(filename=str(FIG_DIR / '04_daily_profile.png')))

### Порівняння робочих і вихідних днів

In [ ]:
display(Image(filename=str(FIG_DIR / '05_workday_weekend_profile.png')))

### Місячна динаміка споживання

In [ ]:
display(Image(filename=str(FIG_DIR / '06_monthly_consumption.png')))

### Heatmap: години × дні тижня

In [ ]:
display(Image(filename=str(FIG_DIR / '07_heatmap_hour_weekday.png')))

### Декомпозиція часового ряду

In [ ]:
display(Image(filename=str(FIG_DIR / '08_trend_decomposition.png')))

### Автокореляція до 168 годин

In [ ]:
display(Image(filename=str(FIG_DIR / '09_autocorrelation.png')))

### Кореляційна матриця факторів

In [ ]:
display(Image(filename=str(FIG_DIR / '10_correlation_matrix.png')))

### Діагностика викидів

In [ ]:
display(Image(filename=str(FIG_DIR / '11_outlier_diagnostics.png')))

### Факт vs прогноз

In [ ]:
display(Image(filename=str(FIG_DIR / '12_actual_vs_prediction.png')))

### Розподіл залишків

In [ ]:
display(Image(filename=str(FIG_DIR / '13_residuals_histogram.png')))

### Важливість ознак

In [ ]:
display(Image(filename=str(FIG_DIR / '14_feature_importance.png')))

### Прогноз на наступний місяць

In [ ]:
display(Image(filename=str(FIG_DIR / '15_next_month_forecast.png')))

## 5. Факторний аналіз

In [ ]:
factor_corr = pd.read_csv(TABLE_DIR / 'factor_correlations.csv')
factor_corr

HDD розраховується як `max(18 - T, 0)`, CDD - як `max(T - 22, 0)`. Найбільший зв’язок зі споживанням мають тарифна зона/ціна, HDD та температура.

## 6. Порівняння моделей прогнозування

In [ ]:
metrics = pd.read_csv(TABLE_DIR / 'model_metrics.csv')
metrics

In [ ]:
best = pd.read_csv(TABLE_DIR / 'best_model.csv')
best

## 7. Прогноз на наступний місяць

In [ ]:
forecast = pd.read_csv(TABLE_DIR / 'forecast_next_month.csv')
forecast.head()

In [ ]:
forecast_summary = forecast.groupby('zone_name', as_index=False).agg(
    forecast_kwh=('forecast_kwh', 'sum'),
    forecast_cost_uah=('forecast_cost_uah', 'sum'),
)
forecast_summary.loc['TOTAL'] = ['TOTAL', forecast_summary['forecast_kwh'].sum(), forecast_summary['forecast_cost_uah'].sum()]
forecast_summary

## 8. Підсумковий висновок

Виконано повний цикл: завантаження даних з ЛР1, EDA, факторний аналіз, feature engineering, побудова моделей, порівняння за R²/RMSE/MAE/MAPE, вибір найкращої моделі та прогнозування наступного місяця з довірчим інтервалом.